In [1]:
import numpy as np 
import matplotlib.pyplot as plt
import tensorflow as tf
import logging 
logging.getLogger("tensorflow").setLevel(logging.ERROR) 
tf.autograph.set_verbosity(0)

In [5]:
def load_coffee_data(): 
    """
        Creates a coffee roasting data set. Roasting duration: 12-15 minutes is beset 
        temperature range: 170 - 260C is best
    """
    rng = np.random.default_rng(2)
    X = rng.random(400).reshape(-1, 2) 
    X[:, 1] = X[:, 1] * 4 + 11.5
    X[:, 0] = X[:, 0] * (285 - 175) + 175

    Y = np.zeros(len(X))
    
    i = 0
    for t, d in X: 
        y = -3/(260 - 175)*t + 21 
        if (t > 175 and t < 260 and d > 12 and d < 15 and d <= y): 
            Y[i] = 1 
        else:  
            Y[i] = 0
        i += 1

    return (X, Y.reshape(-1, 1))

## Dataset

In [6]:
X, Y = load_coffee_data() 
print(X.shape, Y.shape) 

(200, 2) (200, 1)


#### Normalize Data

In [8]:
print(f"Temperature Max, Min pre normalization: {np.max(X[:, 0]):0.2f}, {np.min(X[:, 0]):0.2f}")
print(f"Duration    Max, Min pre normalization: {np.max(X[:, 0]):0.2f}, {np.min(X[:, 0]):0.2f}")
norm_l = tf.keras.layers.Normalization(axis=-1)
norm_l.adapt(X)    # learns mean, variance 
Xn = norm_l(X)

print(f"Temperature Max, Min post normalization: {np.max(Xn[:, 0]):0.2f}, {np.min(Xn[:, 0]):0.2f}")
print(f"Duration    Max, Min pre normalization: {np.max(Xn[:, 0]):0.2f}, {np.min(Xn[:, 0]):0.2f}")

Temperature Max, Min pre normalization: 285.00, 176.08
Duration    Max, Min pre normalization: 285.00, 176.08
Temperature Max, Min post normalization: 1.66, -1.69
Duration    Max, Min pre normalization: 1.66, -1.69


### Numpy Model (Forward Prop in Numpy) 

As described in lecture, it is possible to build your own dense layer using NumPy. This can be utilized to build a multi-layer neural network. 

In the first optional lab, you constructed a neuron in Numpy and in Tensorflow and noted their similarities. A layer contains multiple neurons/units. As described in the lecture, one can utilize a loop for visit each visit (`j`) and sum the bias for the unit (`b[j]`) to form `z`. An activation function (`g(z)`) can then be applied to that result. Let's try below to build a "dense layer" subroutine.

In [9]:
def sigmoid(z): 
    return 1 / (1 + np.exp(-z))

In [10]:
# Define the activation function 
g = sigmoid 

In [31]:
def my_dense(a_in, W, b): 
    """
        Computes dense layer 
        Args: 
            a_in (ndarray(n,)): Data, 1 example 
            W(ndarray((n, j)): Weight matrix, n features per unit, j units 
            b (ndarray(j,)): bias vector, j units 
    """
    units = W.shape[1]
    a_out = np.zeros(units) 
    for j in range(units): 
        w = W[:, j]
        z = np.dot(w, a_in) + b[j]
        a_out[j] = g(z)
    return (a_out)

*Note: You can also implement the function above to accept `g` as an additional parameter. In this notebook though, you will only use one type of activation function (i.e. sigmoid) so it's okay to make it constant and define it outside the function. That's what you did in the code above and it makes the function calls in the next code cells simpler. Just keep in mind that passing it as a parameter is also an acceptable implementation*

The following cell builds a two layer neural network utilizing the `my_dense` subroutine above.

In [13]:
def my_sequential(x, W1, b1, W2, b2): 
    a1 = my_dense(x, W1, b1)
    a2 = my_dense(a1, W2, b2) 
    return (a2)

We can copy trained weights and biases from the previous lab in Tensorflow

In [24]:
W1_tmp = np.array([
    [ 17.966278,   -12.427142,     0.27662078],
    [ 14.914531,    -0.32282764,  11.793835  ]]  )
b1_tmp = np.array([  2.6942885, -13.056967,   14.072244 ])
W2_tmp = np.array( [[ -90.9689  ],
 [-102.874344],
 [  93.43234 ]])
b2_tmp = np.array([-26.530548])

### Predictions 

Once you have a trained model, you can use it to make predictions. Recall that the output of our model is a probability. In this case, the probability of a good roast. To make a decision, one must apply the probability to a threshold. In this case, we will use 0.5

In [32]:
def my_predict(X, W1, b1, W2, b2):
    m = X.shape[0]
    p = np.zeros((m,1))
    for i in range(m):
        p[i,0] = my_sequential(X[i], W1, b1, W2, b2).item()
    return(p)

In [33]:
X_tst = np.array([
    [200,13.9],  # postive example
    [200,17]])   # negative example
X_tstn = norm_l(X_tst)  # remember to normalize
predictions = my_predict(X_tstn, W1_tmp, b1_tmp, W2_tmp, b2_tmp)

In [35]:
yhat = (predictions >= 0.5).astype(int)
print(f"decisions = \n{yhat}")

decisions = 
[[1]
 [0]]


## Network function

This graph shows the operation of the whole network and is identical to the Tensorflow results from the previous lab. The left graph is the raw output of the final layer represented by the blue shading. This is overlaid on the training data represented by the X's and O's. The right graph is the output of the network after a decision threshold. The X's and O's here correspond to decisions made by the network. 